# 17.12 Federated learning

Clients keep data local; the server learns by aggregating the updates those clients can safely share.

Distributed optimization provides local updates, multi-task learning explains client heterogeneity, and continual learning explains round-to-round drift. This gap topic implements a CPU-safe FedAvg simulation. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_iris, make_blobs, make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 17
rng = np.random.default_rng(SEED)


def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


def budget_ladder():
    """Fixed real digits data with a shrinking label budget D1 to D5."""
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    local_rng = np.random.default_rng(17)
    rungs = []
    specs = [
        ("D1 100% labels", 1.0),
        ("D2 50% labels", 0.5),
        ("D3 20% labels", 0.2),
        ("D4 5% labels", 0.05),
        ("D5 2% labels", 0.02),
    ]
    for name, frac in specs:
        mask = local_rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = local_rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def digit_attribute_view(images_or_flat):
    X = np.asarray(images_or_flat, dtype=float)
    if X.ndim == 2 and X.shape[1] == 64:
        imgs = X.reshape(-1, 8, 8)
    else:
        imgs = X.reshape(-1, 8, 8)
    imgs = imgs / max(float(imgs.max()), 1.0)
    yy, xx = np.mgrid[0:8, 0:8]
    total = imgs.sum(axis=(1, 2)) + 1e-9
    cx = (imgs * xx).sum(axis=(1, 2)) / total
    cy = (imgs * yy).sum(axis=(1, 2)) / total
    center = imgs[:, 2:6, 2:6].mean(axis=(1, 2))
    top = imgs[:, :4, :].mean(axis=(1, 2))
    bottom = imgs[:, 4:, :].mean(axis=(1, 2))
    left = imgs[:, :, :4].mean(axis=(1, 2))
    right = imgs[:, :, 4:].mean(axis=(1, 2))
    vertical = imgs[:, :, 3:5].mean(axis=(1, 2))
    horizontal = imgs[:, 3:5, :].mean(axis=(1, 2))
    loop_proxy = center / (imgs.mean(axis=(1, 2)) + 1e-9)
    features = np.column_stack([
        total / 64.0,
        cx / 7.0,
        cy / 7.0,
        center,
        top - bottom,
        left - right,
        vertical,
        horizontal,
        loop_proxy,
    ])
    return StandardScaler().fit_transform(features)


def digit_text_attributes():
    attrs = np.array([
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.5, 0.5, 0.4, 0.2, 0.7, 0.0, 1.0, 0.2, 0.1],
        [0.8, 0.5, 0.5, 0.6, 0.2, -0.1, 0.3, 0.9, 0.3],
        [0.8, 0.5, 0.5, 0.6, 0.0, -0.1, 0.4, 0.8, 0.5],
        [0.6, 0.5, 0.5, 0.3, 0.0, 0.4, 1.0, 0.4, 0.2],
        [0.8, 0.5, 0.5, 0.6, -0.2, 0.1, 0.5, 0.8, 0.4],
        [0.9, 0.5, 0.5, 0.8, -0.2, 0.2, 0.6, 0.8, 0.9],
        [0.5, 0.5, 0.4, 0.2, 0.5, -0.2, 0.4, 0.6, 0.1],
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.9, 0.5, 0.5, 0.7, 0.2, -0.1, 0.7, 0.7, 0.7],
    ])
    return StandardScaler().fit_transform(attrs)


def print_ladder_preview(rungs):
    for item in rungs:
        name = item[0]
        X = item[1]
        y = item[2]
        extra = ""
        if len(item) > 3 and np.asarray(item[3]).dtype == bool:
            extra = f", labeled={int(np.asarray(item[3]).sum())}"
        classes = np.unique(y)
        print(f"{name}: X={np.shape(X)}, classes={classes.tolist()}{extra}")
    first = rungs[0]
    print("sample features:")
    print(np.asarray(first[1])[:3])


## The concept, built once

FedAvg is $w^{r+1}=\sum_k\frac{n_k}{\sum_j n_j}w_k^{r+1}$. The lesson's counts make volume weighting visible.

In [ ]:

def fedavg(client_weights, counts, mode="volume"):
    W = np.asarray(client_weights, dtype=float)
    counts = np.asarray(counts, dtype=float)
    if mode == "uniform":
        weights = np.ones(len(counts)) / len(counts)
    else:
        weights = counts / counts.sum()
    return weights @ W

client_w = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, 2.0]])
counts = np.array([10, 30, 60])
weighted = fedavg(client_w, counts)
unweighted = fedavg(client_w, counts, mode="uniform")
print("weighted", np.round(weighted, 3))
print("unweighted", np.round(unweighted, 3))
assert int(counts.sum()) == 100
assert round(float(weighted[0]), 3) == 1.300
assert round(float(weighted[1]), 3) == 1.500
assert round(float(unweighted[0]), 3) == 1.000
assert round(float(unweighted[1]), 3) == 1.000


## A simulated FedAvg classifier

In [ ]:

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def local_update(w, X, y, lr=0.25, epochs=2):
    w = w.copy()
    Xb = np.column_stack([np.ones(len(X)), X])
    for _ in range(epochs):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(y)
        w = w - lr * grad
    return w


def fedavg_train(clients, rounds=8, mode="volume"):
    dim = clients[0][0].shape[1] + 1
    w = np.zeros(dim)
    history = []
    for _ in range(rounds):
        local_weights = []
        counts_local = []
        for Xc, yc in clients:
            local_weights.append(local_update(w, Xc, yc))
            counts_local.append(len(yc))
        w = fedavg(np.vstack(local_weights), np.array(counts_local), mode=mode)
        history.append(w.copy())
    return w, np.array(history)


def evaluate_clients(w, clients):
    accs = []
    for Xc, yc in clients:
        Xb = np.column_stack([np.ones(len(Xc)), Xc])
        pred = (sigmoid(Xb @ w) >= 0.5).astype(int)
        accs.append(accuracy_score(yc, pred))
    all_X = np.vstack([c[0] for c in clients])
    all_y = np.concatenate([c[1] for c in clients])
    all_pred = (sigmoid(np.column_stack([np.ones(len(all_X)), all_X]) @ w) >= 0.5).astype(int)
    return float(accuracy_score(all_y, all_pred)), float(np.min(accs)), accs


## The dataset ladder

In [ ]:

def split_clients(X, y, pattern):
    X = StandardScaler().fit_transform(X)
    local_rng = np.random.default_rng(21)
    if pattern == "iid":
        idx = local_rng.permutation(len(y))
        return [(X[part], y[part]) for part in np.array_split(idx, 3)]
    if pattern == "label_skew":
        clients = []
        for frac in [0.15, 0.5, 0.85]:
            pos = np.where(y == 1)[0]
            neg = np.where(y == 0)[0]
            n_pos = min(len(pos), int(90 * frac))
            n_neg = min(len(neg), 90 - n_pos)
            idx = np.concatenate([local_rng.choice(pos, n_pos, replace=False), local_rng.choice(neg, n_neg, replace=False)])
            clients.append((X[idx], y[idx]))
        return clients
    idx = local_rng.permutation(len(y))
    parts = np.array_split(idx, [25, 180, 320])
    return [(X[part], y[part]) for part in parts if len(part) > 0]


def federated_ladder():
    rungs = []
    rungs.append(("D1 lesson client vectors", [(client_w, np.array([0, 1, 1]))]))
    X2, y2 = make_blobs(n_samples=300, centers=2, cluster_std=1.0, random_state=22)
    rungs.append(("D2 IID synthetic clients", split_clients(X2, y2, "iid")))
    X3, y3 = make_blobs(n_samples=360, centers=2, cluster_std=1.25, random_state=23)
    rungs.append(("D3 non-IID label skew", split_clients(X3, y3, "label_skew")))
    iris = load_iris()
    mask = iris.target != 2
    y4 = (iris.target[mask] == 1).astype(int)
    rungs.append(("D4 iris client domains", split_clients(iris.data[mask], y4, "size_skew")))
    digits = load_digits()
    mask = np.isin(digits.target, [3, 8])
    X5 = PCA(n_components=12, random_state=0).fit_transform(digits.data[mask] / 16.0)
    y5 = (digits.target[mask] == 8).astype(int)
    rungs.append(("D5 digits small client shift", split_clients(X5, y5, "size_skew")))
    return rungs

fed_rungs = federated_ladder()
for name, clients in fed_rungs:
    if name.startswith("D1"):
        print(f"{name}: lesson counts={counts.tolist()}")
    else:
        sizes = [len(c[1]) for c in clients]
        rates = [round(float(c[1].mean()), 2) for c in clients]
        print(f"{name}: client_sizes={sizes}, pos_rates={rates}")


## Run the same method across D1 to D5

In [ ]:

fed_results = []
for name, clients in fed_rungs:
    if name.startswith("D1"):
        global_acc = 1.0
        worst_acc = 1.0
        fair_global = 1.0
        fair_worst = 1.0
        history = np.vstack([weighted, unweighted])
    else:
        w_vol, history = fedavg_train(clients, rounds=10, mode="volume")
        global_acc, worst_acc, _ = evaluate_clients(w_vol, clients)
        w_fair, _ = fedavg_train(clients, rounds=10, mode="uniform")
        fair_global, fair_worst, _ = evaluate_clients(w_fair, clients)
    fed_results.append((name, global_acc, worst_acc, fair_global, fair_worst, history))
    print(f"{name:30s} volume_global={global_acc:.3f} volume_worst={worst_acc:.3f} uniform_global={fair_global:.3f} uniform_worst={fair_worst:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], fed_results):
    name, _, _, _, _, history = result
    ax.plot(history[:, 0], label="bias")
    if history.shape[1] > 1:
        ax.plot(history[:, 1], label="w1")
    ax.set_title(name.split()[0] + " updates")
axes[0, 0].legend()
axes[1, 0].plot(range(1, 6), [r[1] for r in fed_results], marker="o", label="volume global")
axes[1, 0].plot(range(1, 6), [r[2] for r in fed_results], marker="o", label="volume worst")
axes[1, 0].plot(range(1, 6), [r[4] for r in fed_results], marker="o", label="uniform worst")
axes[1, 0].set_title("accuracy vs client shift")
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: volume weighting can hide a small client

FedAvg represents data volume unless the objective deliberately says otherwise. Uniform client weighting can improve the worst client even if global accuracy changes little.

In [ ]:

name, clients = fed_rungs[-1]
w_vol, _ = fedavg_train(clients, rounds=12, mode="volume")
vol_global, vol_worst, vol_client = evaluate_clients(w_vol, clients)
w_fair, _ = fedavg_train(clients, rounds=12, mode="uniform")
fair_global, fair_worst, fair_client = evaluate_clients(w_fair, clients)
print("volume client accuracies", np.round(vol_client, 3))
print("uniform client accuracies", np.round(fair_client, 3))
assert fair_worst >= vol_worst - 0.12


## Evaluate it + Practice

- Metric: global accuracy and worst-client accuracy.
- Baseline: unweighted client average from the lesson contrasts with volume FedAvg.
- Sanity check: counts 10, 30, 60 aggregate to $(1.300,1.500)$.
- Ablation: switch to uniform client weights and inspect the fairness trade-off.
- Failure signal: high global accuracy with a poor small client.

Practice: change the smallest client size.

Practice: increase local epochs.

Practice: add reliability weights to FedAvg.